In [73]:
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

/tmp/ipykernel_932/3777615979.py:1: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython display
  from IPython.core.display import display, HTML


# 📘 CityAI – Priorització de queixes dels ciutadans de Barcelona

Aquest notebook utilitza un escenari plausible de l'Ajuntament de Barcelona per aplicar els algorismes d'ordenació en un problema real.

## 🧠 Objectius

Aprofundir en l'algoritme d'ordenació de quicksort vist a teoria, i veure la seva aplicació en un cas real.

## 🧩 Context

L'Ajuntament de Barcelona recull les queixes dels ciutadans al llarg de l'any i les classifica pel seu tipus, per l'àrea de l'ajuntament que les ha de resoldre, per la data i pel lloc on s'han produit.

Aquest cop agafarem les dades originals i amb Python les processarem per a convertir-les en un diccionari manejable. Això ho farem amb l'ajut d'una llibreria, pandas, que veureu en cursos més avançats i per això aquesta part ja us la donem feta.

In [90]:
from collections.abc import Callable
from typing import Any

In [117]:
def print_queixa_format(q: list[Any], target: str) -> None:
    if target == "all":
        for i in q:
            print(i)
    else:
        for i in q:
            print(i[target])

    print()

In [75]:
import pandas as pd

path = "2025_IRIS_Peticions_Ciutadanes_OpenData.csv"
df = pd.read_csv(path)
df.head()

peticions = []

for _, row in df.iterrows():
    peticions.append({
        "id": row["FITXA_ID"],
        "tipus": str(row["TIPUS"]).replace('"', ''),
        "area": str(row["AREA"]).replace('"', ''),
        "detall": str(row["DETALL"]).replace('"', ''),
        "dia": int(row["DIA_DATA_ALTA"]),
        "mes": int(row["MES_DATA_ALTA"]),
        "any": int(row["ANY_DATA_ALTA"]),
        "districte": str(row["DISTRICTE"]).replace('"', ''),
        "barri": str(row["BARRI"]).replace('"', '')
    })

len(peticions)

58932

In [76]:
print(peticions[450:455])

[{'id': 38766075, 'tipus': 'INCIDENCIA', 'area': "Recollida i neteja de l'espai urbà", 'detall': 'Objectes a netejar / retirar', 'dia': 31, 'mes': 12, 'any': 2024, 'districte': 'Sant Martí', 'barri': 'Diagonal Mar i el Front Marítim del Poblenou'}, {'id': 38766081, 'tipus': 'SUGGERIMENT', 'area': "Manteniment de l'espai urbà", 'detall': "Agricultura urbana i col.laboració ciutadana 'mans al verd'", 'dia': 31, 'mes': 12, 'any': 2024, 'districte': 'Sant Andreu', 'barri': 'Sant Andreu'}, {'id': 38766129, 'tipus': 'INCIDENCIA', 'area': "Recollida i neteja de l'espai urbà", 'detall': 'Objectes a netejar / retirar', 'dia': 31, 'mes': 12, 'any': 2024, 'districte': 'Eixample', 'barri': 'Sant Antoni'}, {'id': 38765948, 'tipus': 'INCIDENCIA', 'area': 'Mobilitat', 'detall': 'Bicicleta abandonada', 'dia': 31, 'mes': 12, 'any': 2024, 'districte': 'Gràcia', 'barri': 'la Vila de Gràcia'}, {'id': 38766101, 'tipus': 'CONSULTA', 'area': 'Mobilitat', 'detall': 'Aparcaments al carrer (àrea verda  àrea bla

## 🧠 Funcions lambda

Recorda que podem usar una funció lambda com a paràmetre per a especificar un criteri d'ordenació en algoritmes d'ordenació ja implementats a python, com en aquest exemple:

### ✔️ Exemple per ordenar

In [77]:
fruites = ["poma", "plàtan", "maduixa"]
sorted(fruites, key=lambda x: len(x))

['poma', 'plàtan', 'maduixa']

La gràcia de les funcions lambda és que es poden passar per paràmetre també. 

## ✍️ Exercici 1: Adaptació de quicksort a ordenació per criteris diferents amb funcions lambda

Adapta el codi del quicksort amb una variació: podrem usar una funció lambda "key" com a criteri d'ordenació.

A continuació us copiem el codi original de quicksort, cal que l'adapteu per a poder ordenar per qualsevol criteri que vulgui l'usuari mitjançant funcions lambda.

### Codi original de quicksort

In [78]:
# Codi original de la funció partició de quicksort

def particio(
    llista:list[any], primer: int, darrer: int
) -> int:
    """Posa el pivot en la seva posició ordenada dins un tros de llista (slice).

    Aquesta funció considera els elements de la llista compresos entre el primer
    i el darrer índexs indicats, defineix el primer com el pivot, i el situa en
    la posició que tindria dins d'aquest tros de llista si els elements
    estiguessin ordenats. Tots els elements amb valors més petits obtenen
    posicions amb índexs més baixos que el pivot i els que tenen valors més grans
    obtenen índexs més alts, tot i que aquests elements més petits i més grans no
    tenen perquè estar ordenats entre sí. Els elements de la llista amb índexs
    més baixos que el pivot no es modifiquen. Aquí s'implementa l'algoritme de
    partició de Hoare, que és el més ràpid.

    En l'algoritme d'ordenació ràpida o quicksort, aquest pas de partició es pot
    repetir diverses vegades i obtenir una llista ordenada.

    Aquesta funció no genera una llista nova sinó que modifica la d'entrada i
    retorna l'índex que tindria el pivot en la llista si el tros de llista
    especificat estigués ordenat.

    Paràmetres:
    -----------
    llista (list): Llista d'elements per a ordenar.
    primer (int): Primer índex del tros de la llista que es té en compte (slice),
        i també índex del pivot o tupla de referència que es situarà en la seva
        posició ordenada dins d'aquest tros de llista.
    darrer (int): Darrer índex del tros de la llista que es té en compte.

    Retorna:
    --------
    Índex que tindia l'element que fa de pivot en la llista si s'ordenés aquest tros.
    """
    pivot: int = primer
    i: int = primer + 1
    j: int = darrer
    indexs_creuats: bool = False

    # Recorre el tros de llista especificat començant tant pel principi com pel 
    # final fins que els dos índexs es troben
    while not(indexs_creuats):

        # Augmenta l'índex i des del principi de la subllista fins que troba un 
        # element fora de lloc (més gran que el pivot)
        while i <= darrer and llista[i] <= llista[pivot]:
            i += 1
        
        # Disminueix l'índex j des del final de la subllista fins que troba un
        # element fora de lloc o el pivot (més petit o igual que el pivot)
        while j >= primer and llista[j] > llista[pivot]:
            j -= 1

        # Mentre els dos índexs no es trobin, intercanvia els elements trobats 
        # pels dos índexs. Quan els dos índexs es troben o es creuen,
        # intercanvia el pivot amb aquest element i atura el bucle.
        if i >= j: 
            indexs_creuats = True
        else: 
            llista[i], llista[j] = llista[j], llista[i]

    llista[j], llista[pivot] = llista[pivot], llista[j]
    return j

In [79]:
# Codi original de la funció recursiva de quicksort

def quicksort_r(
    llista: list[any], primer: int, darrer: int
) -> None:
    """
    Partició recursiva de les llistes a cada banda del pivot (quicksort).

    Aquesta funció implementa un algoritme recursiu de quicksort triant sempre
    com a pivot el primer element de la llista. Un cop ha trobat la posició del
    pivot, torna a executar aquesta mateixa funció pel tros de la llista amb
    valors més petits que pivot i pel tros de la llista amb valors més grans al
    pivot, fins que el pivot és l'únic element.
    
    Paràmetres:
    -----------
    llista (llista): Llista a ordenar.
    primer (int): Primer índex del tros de la llista que es té en compte (slice),
        i també índex del pivot LaTeXque es situarà en la seva posició ordenada dins
        d'aquest tros de llista.
    darrer (int): Darrer índex del tros de la llista que es té en compte.

    Retorna:
    --------
    None
    """
    # Només segueix executant la funció recursivament si li queden al menys dos
    # elements a la llista
    if darrer > primer:
        
        # Executa la partició per a trobar la posició del pivot a la llista
        pivot: int = particio(llista, primer, darrer)

        # Un cop sap la posició del pivot, executa aquesta mateixa funció per la
        # subllista de valors més petits i la de valors més grans que el pivot
        quicksort_r(llista, primer, pivot - 1) 
        quicksort_r(llista, pivot + 1, darrer) 

In [80]:
# Codi original de la funció de quicksort

def quicksort(llista: list[any]) -> list[any]:
    """Ordena una llista d'elements usant l'algoritme quicksort.
    
    Aquesta funció ordena en ordre ascendent una llista d'elements, tot usant
    l'algoritme d'ordenació de quicksort.
    
    Paràmetres
    -----------
        llista: Llista a ordenar.

    Retorna
    -------
        Llista ordenada en ordre ascendent.
    """
    # Fem l'execució inicial de la funció recursiva usant la llista sencera.
    quicksort_r(llista, 0, len(llista) - 1)
    return llista

### Codi adaptat quicksort 
Introduïu a continuació el codi adaptat de l'algoritme de quicksort.

In [81]:
# Adapta el codi de la funció partició de quicksort a l'ús de qualsevol criteri d'ordenació (funció lambda)

def particio_criteris(llista: list[Any], left: int, right: int, key: Callable[[Any], Any]) -> int:
    """Ordena el pivot a la llista segons el criteri especificat.

    Aquesta funció considera els elements de la llista compresos entre el primer
    i el darrer índexs indicats, defineix el primer com el pivot, i el situa en
    la posició que tindria dins d'aquest tros de llista si els elements
    estiguessin ordenats segons el criteri especificat per l'usuari. Tots els
    elements amb valors més petits segons el criteri especificat obtenen posicions
    amb índexs més baixos que el pivot i els que tenen valors més grans obtenen
    índexs més alts, tot i que aquests elements més petits i més grans no tenen 
    perquè estar ordenats entre sí. Els elements de la llista amb índexs
    més baixos que el pivot no es modifiquen. Aquí s'implementa l'algoritme de
    partició de Hoare, que és el més ràpid.

    En l'algoritme d'ordenació ràpida o quicksort, aquest pas de partició es pot
    repetir diverses vegades i obtenir una llista ordenada.

    Aquesta funció no genera una llista nova sinó que modifica la d'entrada i
    retorna l'índex que tindria el pivot en la llista si el tros de llista
    especificat estigués ordenat.
    
    Paràmetres
    ----------
    [Omple tu els paràmetres]
    
    Retorna
    -------
    Índex que tindia l'element que fa de pivot en la llista si s'ordenés aquest tros.
    """

    i = left - 1
    j = right + 1
    pivot_key = key(llista[left])
    
    while True:
        i += 1
        while key(llista[i]) < pivot_key:
            i += 1
        
        j -= 1
        while key(llista[j]) > pivot_key:
            j -= 1
        
        if i >= j: return j
        llista[i], llista[j] = llista[j], llista[i]

In [82]:
# Adapta el codi de la funció recursiva de quicksort a l'ús de qualsevol criteri d'ordenació (funció lambda)

def quicksort_criteris_r(llista: list[int], left: int, right: int, key: Callable[[Any], Any]) -> None:
    """
    Partició recursiva amb el criteri especificat usant l'algoritme quicksort.

    Aquesta funció implementa un algoritme recursiu de quicksort, triant sempre
    com a pivot el primer element de la llista, on l'usuari pot especificar un
    criteri d'ordenació amb una funció lambda. Un cop ha trobat la posició del
    pivot, torna a executar aquesta mateixa funció pel tros de la llista amb 
    valors més petits que pivot i pel tros de la llista amb valors més grans al 
    pivot, fins que el pivot és l'únic element.
    
    Paràmetres
    ----------
    [Omple tu els paràmetres]

    Retorna
    -------
    None
    """
    if left >= right:
        return
    
    pivot = particio_criteris(llista, left, right, key)
    quicksort_criteris_r(llista, left, pivot, key)
    quicksort_criteris_r(llista, pivot+1, right, key)
    

In [83]:
# Adapta el codi de la funció de quicksort a l'ús de qualsevol criteri d'ordenació (funció lambda)

def quicksort_criteris(llista: list[int], key: Callable[[Any], Any]) -> list:
    """Ordena una llista d'elements pel criteri especificat usant l'algoritme quicksort.

    Aquesta funció ordena una llista d'elements pel criteri especificat amb la funció
    lambda, mitjançant l'algoritme d'ordenació de quicksort.
    
    Paràmetres
    -----------
    [Omple tu els paràmetres.]

    Retorna
    -------
    Llista ordenada amb el criteri especificat.
    """
    quicksort_criteris_r(llista, 0, len(llista)-1, key)
    return llista

Farem una primera prova en la que la funció key serà l'identificador de petició.

In [84]:
key_id = lambda p: p["id"]

# No ordeno totes les peticions perquè trigaria massa, ordeno les primeres 2000 peticions
ordenades_id = quicksort_criteris(peticions[:2000], key_id)
print(type(ordenades_id))
for i in ordenades_id:
    print(i["id"])
# print(ordenades_id[:5])

<class 'list'>
38755903
38755904
38755905
38755906
38755907
38755908
38755909
38755910
38755911
38755912
38755913
38755914
38755915
38755916
38755917
38755918
38755919
38755920
38755921
38755922
38755923
38755924
38755925
38755926
38755927
38755928
38755929
38755930
38755931
38755932
38755933
38755934
38755935
38755936
38755937
38755938
38755939
38755940
38755941
38755942
38755943
38755944
38755945
38755946
38755947
38755948
38755949
38755950
38755951
38755952
38755953
38755954
38755955
38755956
38755957
38755958
38755959
38755960
38755961
38755962
38755963
38755964
38755965
38755966
38755967
38755968
38755969
38755970
38755971
38755972
38755973
38755974
38755975
38755976
38755977
38755978
38755979
38755980
38755981
38755982
38755983
38755984
38755985
38755986
38755987
38755988
38755989
38755990
38755991
38755992
38755993
38755994
38755995
38755996
38755997
38755998
38755999
38756000
38756001
38756002
38756003
38756004
38756005
38756006
38756007
38756008
38756009
38756010
38756011
3875

## ✍️ Exercici 2: Explora diferents criteris d'ordenació

Crea les funcions lambda per establir els següents criteris d'ordenació:

1. Ordenar per les últimes lletres del barri (és a dir, en comptes de començar a ordenar per les lletres del principi, que ordeni per les lletres del final)
2. Ordenar pel seu id però a la inversa (és a dir, en ordre descendent)
3. Ordenar per data (any, mes i dia)

In [94]:
# Completa les funcions lambda
key_barri_ultimes = lambda q : q["barri"][::-1]
key_id_invers = lambda q : -q["id"]
key_data = lambda q : (q["any"], q["mes"], q["dia"])

In [114]:
print(peticions[-1]["barri"], " | ", key_barri_ultimes(peticions[-1]))
print(peticions[-1]["id"], " | ", key_id_invers(peticions[-1]))


el Poblenou  |  uonelboP le
40467549  |  -40467549


In [118]:
# Comprovo que ordena els diferents elements usant el criteri que hem especificat

ordenades_barri_ultimes = quicksort_criteris(peticions[:2000], key_barri_ultimes)
print_queixa_format(ordenades_barri_ultimes[:3], "barri"); 
print_queixa_format(ordenades_barri_ultimes[-3:], "barri")

ordenades_id_invers = quicksort_criteris(peticions[:2000], key_id_invers)
print_queixa_format(ordenades_id_invers[:3], "id"); 
print_queixa_format(ordenades_id_invers[-3:], "id")

ordenades_data = quicksort_criteris(peticions[:2000], key_data)
# print(ordenades_data[:3], ordenades_data[-3:])  
print_queixa_format(ordenades_data[:3], "all"); 
print_queixa_format(ordenades_data[-3:], "all")





el Putxet i el Farró
el Putxet i el Farró
el Putxet i el Farró

38784001
38783983
38783982

38755905
38755904
38755903

{'id': 38757087, 'tipus': 'SUGGERIMENT', 'area': 'Cultura', 'detall': 'Altres equipaments suggeriments', 'dia': 7, 'mes': 5, 'any': 2024, 'districte': 'nan', 'barri': 'nan'}
{'id': 38755987, 'tipus': 'QUEIXA', 'area': 'Serveis socials', 'detall': 'Desacord amb la prestació de serveis', 'dia': 4, 'mes': 7, 'any': 2024, 'districte': 'nan', 'barri': 'nan'}
{'id': 38755996, 'tipus': 'QUEIXA', 'area': 'Serveis socials', 'detall': 'Desacord amb la prestació de serveis', 'dia': 9, 'mes': 7, 'any': 2024, 'districte': 'nan', 'barri': 'nan'}

{'id': 38776973, 'tipus': 'INCIDENCIA', 'area': "Recollida i neteja de l'espai urbà", 'detall': 'Objectes a netejar / retirar', 'dia': 6, 'mes': 1, 'any': 2025, 'districte': 'Sants-Montjuïc', 'barri': 'el Poble-sec'}
{'id': 38776965, 'tipus': 'PETICIO DE SERVEI', 'area': 'Gestions municipals', 'detall': 'Rec. animals morts en espais pr

## ✍️ Exercici 3: Calcula la complexitat temporal del quicksort

Calcula la complexitat temporal empírica de la teva implementació del quicksort. Fes-ho usant la comanda %%timeit en una cel·la del Jupyter notebook, mesurant què triga en el cas d'ordenar per diferents criteris, per exemple, tot executant en una cel·la separada la comanda "ordenades_data = quicksort_criteris(peticions, key_data)". No afegeixis cap print ni altres coses que puguin afegir complexitat temporal i emmascarar la complexitat del teu algoritme. Si veus que triga massa, fes l'anàlisi només amb les 2000 primeres peticions.

#### Mesura de la complexitat de quicksort amb els 5 criteris

In [146]:
key_data = lambda q : (q["any"], q["mes"], q["dia"])

In [147]:
# Mesura en aquesta cel·la la complexitat temporal de quicksort usant el criteri de la data
%timeit quicksort_criteris(peticions[:2000].copy(), key_data)

100 ms ± 945 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [129]:
key_id_invers = lambda q : -q["id"]

In [130]:
# Mesura en aquesta cel·la la complexitat temporal de quicksort usant el criteri de l'id en ordre descendent
%timeit quicksort_criteris(peticions[:2000].copy(), key_id_invers)

2.22 ms ± 58.3 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [131]:
key_llargada_detall = lambda q : len(q["detall"])

In [132]:
# Mesura en aquesta cel·la la complexitat temporal de quicksort usant el criteri de la llargada del detall
%timeit quicksort_criteris(peticions[:2000].copy(), key_llargada_detall)

1.89 ms ± 15.6 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [135]:
key_districte = lambda q : q["districte"]

In [136]:
# Mesura en aquesta cel·la la complexitat temporal de quicksort usant el criteri del districte
%timeit quicksort_criteris(peticions[:2000].copy(), key_districte)

1.9 ms ± 23.4 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [137]:
key_barri_ultimes = lambda q : q["barri"][::-1]

In [138]:
# Mesura en aquesta cel·la la complexitat temporal de quicksort usant el criteri de les últimes lletres del barri
%timeit quicksort_criteris(peticions[:2000].copy(), key_barri_ultimes)

3.34 ms ± 55.4 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


#### Mesura empírica de complexitat del quicksort:

| Criteri | Quicksort |
|------|-----------------|
| Data | 100 ms ± 945 µs |
| Id (invers) | 2.22 ms ± 58.3 µs |
| Detall (llargada) | 1.89 ms ± 15.6 µs |
| Districte | 1.9 ms ± 23.4 µs |
| Barri (últimes) | 3.34 ms ± 55.4 µs|

## ✍️ Exercici 4: Compara la complexitat temporal del quicksort amb la del mergesort

Copia aquí les funcions de mergesort que vas implementar a la sessió anterior (Bloc9 CityAI), i compara la complexitat temporal dels algoritmes de quicksort i mergesort, tot mesurant què triga en ordenar la llista pels mateixos criteris.

Quin algoritme té menys complexitat temporal? Hi ha molta diferència o són similars? Depèn del criteri pel que ordenem o hi ha un algoritme que sigui més eficient amb tots els criteris d'ordenació?

In [200]:
def merge_sort(l: list[dict], key: Callable) -> list[dict]:
    merge_sort_r(l, 0, len(l)-1, key)
    return l

In [201]:
def merge_sort_r(l: list[dict], left: int, right: int, key: Callable) -> None:
    if (left >= right) : return

    mid = (left + right) // 2
    merge_sort_r(l, left, mid, key)
    merge_sort_r(l, mid+1, right, key)

    merge(l, left, mid, right, key)

In [208]:
def merge(l: list[dict[str, Any]], left: int, mid: int, right: int, key: Callable) -> None:
    aux = []
    # print(l[left:right+1], aux, end=" ")

    i = left; j = mid+1
    ki = key(l[i]); kj = key(l[j])
    while i <= mid and j <= right:
        ki = key(l[i]); kj = key(l[j])

        if ki <= kj:
            aux.append(l[i])
            i += 1
        else:
            aux.append(l[j])
            j += 1

    aux.extend(l[i : mid+1])
    aux.extend(l[j : right+1])

    # print(aux)

    l[left:right+1] = aux


#### Mesura de la complexitat del mergesort amb els 5 criteris

In [209]:
key_data = lambda q : (q["any"], q["mes"], q["dia"])

In [210]:
# Mesura en aquesta cel·la la complexitat temporal de mergesort usant el criteri de la data
%timeit merge_sort(peticions[:2000].copy(), key_data)

2.74 ms ± 107 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [211]:
key_id_invers = lambda q : -q["id"]

In [212]:
# Mesura en aquesta cel·la la complexitat temporal de mergesort usant el criteri de l'id en ordre descendent
%timeit merge_sort(peticions[:2000].copy(), key_id_invers)

2.89 ms ± 27.9 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [213]:
key_llargada_detall = lambda q : len(q["detall"])

In [215]:
# Mesura en aquesta cel·la la complexitat temporal de mergesort usant el criteri de la llargada del detall
%timeit merge_sort(peticions[:2000].copy(), key_llargada_detall)

3.05 ms ± 89.6 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [216]:
key_districte = lambda q : q["districte"]

In [218]:
# Mesura en aquesta cel·la la complexitat temporal de mergesort usant el criteri del districte
%timeit merge_sort(peticions[:2000].copy(), key_districte)

2.66 ms ± 52 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [ ]:
key_barri_ultimes = lambda q : q["barri"][::-1]

In [219]:
# Mesura en aquesta cel·la la complexitat temporal de mergesort usant el criteri de les últimes lletres del barri
%timeit merge_sort(peticions[:2000].copy(), key_barri_ultimes)


5.3 ms ± 69.3 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


#### Comparació entre les complexitats de quicksort i mergesort:

| Criteri   | Quicksort | Mergesort |
|-----------|-----------|-----------|
| Data              | 100 ms ± 945 µs | 2.74 ms ± 107 µs |
| Id (invers)       | 2.22 ms ± 58.3 µs | 2.89 ms ± 27.9 µs |
| Detall (llargada) | 1.89 ms ± 15.6 µs | 3.05 ms ± 89.6 µs |
| Districte         | 1.9 ms ± 23.4 µs | 2.66 ms ± 52 µs |
| Barri (últimes)   | 3.34 ms ± 55.4 µs | 5.3 ms ± 69.3 µs |

